In [3]:
import streamlit as st
import google.generativeai as genai
import pandas as pd
import os
from IPython.display import display, Markdown
from dotenv import load_dotenv

load_dotenv() 
API_KEY = os.getenv("gemini_API_KEY") 
genai.configure(api_key=API_KEY)
llm_model = genai.GenerativeModel('gemini-2.5-flash-lite')



In [ ]:
process_path = 'Process_Data'
reorder_df = pd.read_csv(os.path.join(process_path, 'reorder_recommendations.csv'))
mba_rules_df = pd.read_csv(os.path.join(process_path, 'market_basket.csv'))


In [28]:
def get_reorder_predictions(user_id_input, top_n=3):
    try:
        uid = int(user_id_input) 
    except ValueError:
        return []

    user_data = reorder_df[reorder_df['user_id'] == uid]
    
    if user_data.empty:
        return []
        
    top_items = user_data.sort_values(by='rank').head(top_n)
    
    return top_items['product_name'].tolist()

In [29]:
def get_mba_recommendations(product_a, top_n=2):
    rules = mba_rules_df[mba_rules_df['product_name_A'].str.lower() == product_a.lower()]
    
    if rules.empty:
         return []
         
    sorted_rules = rules.sort_values(by='lift', ascending=False).head(top_n)
    
    recs = []
    for _, row in sorted_rules.iterrows():
        recs.append(f"{row['product_name_B']} (ความมั่นใจ {row['confidence']:.2f})")
    return recs

In [30]:
print("Starting Instacart AI Agent!")
print("Type 'q' or 'exit' to quit.\n")

while True:
    # An input box will appear below the cell
    prompt = input("You (Enter Customer ID or Product Name): ")
    
    if prompt.lower() in ['q', 'exit', 'quit']:
        print("Ending conversation.")
        break
        
    if not prompt.strip():
        continue
        
    ai_prompt = ""

    # If input is Customer ID
    if prompt.isdigit(): 
        user_id = prompt
        reorder_items = get_reorder_predictions(user_id)
        
        if reorder_items:
            main_item = reorder_items[0]
            cross_sell_items = get_mba_recommendations(main_item)
            
            ai_prompt = f"""
            You are an AI Marketing Assistant.
            Customer ID: {user_id}
            1. Frequently reordered items: {', '.join(reorder_items)}
            2. Recommended cross-sell items for '{main_item}': {', '.join(cross_sell_items) if cross_sell_items else 'None'}
            
            Task: Write a short, friendly notification message to remind the customer that their items might be running out and subtly offer a cross-sell promotion.
            """
        else:
            print(f"AI: No purchase history found for Customer ID {user_id}.\n")
            continue

    # If input is Product Name
    else:
        product_name = prompt
        cross_sell_items = get_mba_recommendations(product_name)
        
        if cross_sell_items:
            ai_prompt = f"""
            Main Product: '{product_name}' 
            Frequently bought together (MBA): {', '.join(cross_sell_items)}
            
            Task: Briefly explain why these items go well together and suggest a paired promotion idea. 
            """
        else:
            print(f"AI: No Association Rules data available for '{product_name}'.\n")
            continue

    # Call AI
    try:
        response = llm_model.generate_content(ai_prompt)
        print("AI is thinking...\n")
        display(Markdown(f"**AI Response:**\n{response.text}"))
        print("-" * 50)
    except Exception as e:
        print(f"Error: {e}\n")

Starting Instacart AI Agent!
Type 'q' or 'exit' to quit.

AI: No purchase history found for Customer ID 1000001.

AI: No purchase history found for Customer ID 100001.

AI is thinking...



**AI Response:**
Subject: Time to Restock Your Favorites? 🍌🍓

Hi there, Customer ID 10001!

Just a friendly heads-up that your go-to items like Bananas and Kombucha might be running low soon.

While you're thinking about your next order, have you considered pairing your bananas with some **Simply... Go-Gurt Strawberry** or **Simply... Go-Gurt Mixed Berry**? They're a delicious and convenient snack option!

Happy shopping!

--------------------------------------------------
AI is thinking...



**AI Response:**
Subject: Psst! Time for a refresh? 🍕🍋🍫

Hey there, Customer ID 100005!

Just a friendly reminder that your favorites might be running low! We noticed you often reorder our delicious **Classic Uncured Pepperoni Pizza**, refreshing **Sparkling Lemon Water**, and tasty **Chocolate Cheerios Cereal**.

Wouldn't it be great to have everything you need on hand? Let us know if you'd like to add any of these to your next order!

Happy shopping!

--------------------------------------------------
AI: No Association Rules data available for 'Go-Gurt Mixed Berry'.

AI: No Association Rules data available for 'Chocolate Cheerios Cereal.'.

AI is thinking...



**AI Response:**
These items go well together because they both offer **convenient and healthy snack options**, particularly appealing for on-the-go consumption or for children's lunches. Bananas provide natural sweetness and energy, while Go-Gurt offers a creamy, protein-rich complement that can be a fun and appealing way for kids (and adults!) to get their dairy.

**Paired Promotion Idea:**

**"Banana Berry Snack Pack!"**

*   **Offer:** Bundle a bunch of bananas with a multi-pack of Simply Go-Gurt (either Strawberry, Mixed Berry, or a variety pack).
*   **Messaging:** "Fuel your day the delicious way! Grab a Banana Berry Snack Pack for a perfect on-the-go combination of natural energy and creamy goodness. Great for lunchboxes, after-school snacks, or anytime you need a healthy boost!"
*   **Visuals:** Feature an image of a banana next to a colorful Go-Gurt pouch, perhaps with kids enjoying them.
*   **Possible Extension:** Offer a small discount when purchasing the bundle, or a branded reusable snack bag with purchase.

--------------------------------------------------
AI: No Association Rules data available for 'Simply Go-Gurt'.

AI is thinking...



**AI Response:**
Here's a breakdown of the MBA items and a promotion idea:

**Why they go well together:**

*   **Carrots & Turnips:** Both are root vegetables that are often roasted, stewed, or used in mirepoix (flavor base for soups and stews). They offer complementary earthy flavors and a satisfying sweetness from the carrots when cooked.
*   **Carrots & Garbanzo Beans:** Garbanzo beans (chickpeas) are a good source of protein and fiber. When combined with the sweetness and texture of carrots, they create a more complete and hearty dish, especially in vegetarian or vegan meals.

**Paired Promotion Idea:**

**"Root Vegetable & Bean Medley Kit"**

*   **Concept:** Offer a bundle that includes a bag of carrots, a can of no-salt-added garbanzo beans, and a bunch of purple top turnips.
*   **Promotional Messaging:** "Effortless flavor for your next healthy meal! Our Root Vegetable & Bean Medley Kit makes it easy to create delicious roasted dishes, hearty stews, or satisfying salads. Packed with nutrients and natural sweetness, these ingredients are a match made in culinary heaven."
*   **In-Store/Online Display:**
    *   **Recipe Card:** Include a simple recipe card showcasing a dish using all three ingredients (e.g., "Roasted Root Vegetable & Chickpea Salad" or "Hearty Vegetable Stew").
    *   **Visual Appeal:** Display the items together in a visually appealing way, perhaps with a small chalkboard sign highlighting the "Medley Kit."
    *   **Cross-Promotion:** Place the kit near other complementary ingredients like olive oil, herbs, or vegetable broth.

This promotion leverages the idea that these items can be used together to create a complete and healthy meal, making shopping more convenient for the customer.

--------------------------------------------------
AI: No Association Rules data available for 'Escape'.

Ending conversation.
